# LangChain + Groq Revision Notes

**Goal:** revise the concepts behind the uploaded scripts in one connected teaching flow, not file-by-file.

The scripts together are about building small LLM applications using **LangChain + Groq**:

1. setting up a chat model,
2. creating prompts,
3. sending normal messages,
4. batching many prompts,
5. streaming partial responses,
6. reading async streaming events,
7. caching responses,
8. tracking token usage,
9. forcing structured output using schemas and Pydantic.

Use this as a **study + interview revision notebook**. Read each concept, then run the code cell below it.

---

## 0. Big Picture Mental Model

A simple LangChain LLM app has this pipeline:

```text
user input
   ↓
prompt / messages
   ↓
chat model wrapper: ChatGroq
   ↓
AIMessage response
   ↓
content / metadata / token usage / structured data
```

The important idea is this:

> LangChain does not give the LLM magical memory or internet access. It gives you clean Python objects and methods for sending prompts, receiving responses, streaming, batching, caching, parsing, and composing pipelines.

---

## 1. Setup: Environment, API Key, and Model

### What we are building

We create one reusable `ChatGroq` model object. Every later example uses this object.

### Background idea

Groq is the provider running the LLM. LangChain is the Python interface around it.

`ChatGroq(...)` is a **chat model wrapper**. It behaves like a LangChain `Runnable`, meaning it supports methods like:

- `.invoke()`
- `.batch()`
- `.stream()`
- `.astream_events()`

### Input and output

Input can be:

- a plain string,
- a prompt object,
- a list of chat messages.

Output is usually an `AIMessage`, not a plain string.

Important fields:

| Field | Meaning |
|---|---|
| `response.content` | final text answer |
| `response.usage_metadata` | normalized token counts |
| `response.response_metadata` | provider-specific metadata such as finish reason, timings, raw token usage |

### Before running

Create a `.env` file in your notebook/script folder:

```bash
GROQ_API_KEY=your_key_here
```

Then install packages:

```bash
pip install langchain langchain-groq langchain-community python-dotenv pydantic
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_groq import ChatGroq

# Prefer GROQ_API_KEY. Some scripts used MY_CUSTOM_API_KEY, so this supports both.
api_key = os.getenv("GROQ_API_KEY") or os.getenv("MY_CUSTOM_API_KEY")

if api_key is None:
    print("No API key found. Add GROQ_API_KEY to your .env file before running model calls.")
    model = None
else:
    model = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=api_key,
        temperature=0,
    )
    print("Model ready.")


### Function-by-function explanation

There is no custom function yet, but understand these methods:

| Method / class | Purpose |
|---|---|
| `load_dotenv()` | Loads environment variables from a `.env` file into Python's environment. |
| `os.getenv("GROQ_API_KEY")` | Reads the API key safely without hardcoding it. |
| `ChatGroq(...)` | Creates a LangChain chat model object connected to Groq. |
| `temperature=0` | Makes output more deterministic. Useful for factual answers, extraction, and structured output. |

### Common setup errors

| Error | Cause | Fix |
|---|---|---|
| API key error | `.env` missing or wrong variable name | Use `GROQ_API_KEY=...` |
| `ModuleNotFoundError` | package not installed | `pip install langchain-groq python-dotenv` |
| model name error | wrong model string | check Groq model name spelling |

---

## 2. Basic Model Call: `.invoke()`

### What we are building

We send one prompt to the model and print the answer.

### Background idea

`.invoke()` means:

> Take one input, run the model once, return one complete response.

It is **blocking**, meaning Python waits until the full answer is complete.

### Input and output

Input:

```python
"Where is the Statue of Liberty?"
```

Output:

```python
AIMessage(...)
```

To get only the text:

```python
response.content
```


In [ ]:
if model:
    response = model.invoke("Where is the Statue of Liberty?")

    print("TEXT OUTPUT:")
    print(response.content)

    print("\nOBJECT TYPE:")
    print(type(response))

    print("\nNORMALIZED TOKEN USAGE:")
    print(response.usage_metadata)

    print("\nFINISH REASON:")
    print(response.response_metadata.get("finish_reason"))


### What happens internally

1. LangChain converts your input into a chat format.
2. It sends the request to Groq.
3. Groq runs the model and returns output text plus metadata.
4. LangChain wraps that into an `AIMessage` object.

### Important interview point

`.invoke()` does **not** return only a string because production apps need more than text:

- token usage,
- finish reason,
- timing,
- provider metadata,
- tool-call information,
- response IDs.

---

## 3. Chat Configuration: `temperature`, `max_tokens`, and `api_key`

### What we are building

We create different model objects with different behavior.

### Background idea

A model object is not just the model name. It also stores generation settings.

| Setting | Meaning | Good for |
|---|---|---|
| `temperature=0` | less random, more deterministic | extraction, structured output, factual tasks |
| `temperature=0.7` | more variation | creative writing, brainstorming |
| `max_tokens=55` | maximum output tokens | controlling answer length |
| `api_key=...` | explicit API key | custom key names or multiple providers |

### Important warning

`max_tokens` controls **output length**, not input length. If it is too small, the model may stop mid-answer.


In [ ]:
if api_key:
    short_model = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=api_key,
        temperature=0,
        max_tokens=40,
    )

    response = short_model.invoke("Explain quantum physics in a simple way.")
    print(response.content)
    print("finish_reason:", response.response_metadata.get("finish_reason"))


### Sanity check

If the answer looks cut off, check:

```python
response.response_metadata.get("finish_reason")
```

Possible values usually include:

| Finish reason | Meaning |
|---|---|
| `stop` | completed normally |
| `length` | stopped because token limit was reached |

### Common error

Do not set `max_tokens` too low for explanation tasks. Use low values only when you intentionally need short answers.

---

## 4. Prompt Templates: `PromptTemplate`

### What we are building

A reusable prompt with placeholders.

Instead of writing this repeatedly:

```python
"What is the capital of India?"
"What is the capital of Japan?"
"What is the capital of Brazil?"
```

we create a template:

```python
"What is the capital of {country}?"
```

### Background idea

A prompt template separates:

- the fixed instruction, and
- the changing input values.

This is important in real apps because user input, retrieved documents, parameters, or metadata can be inserted dynamically.

### Input and output format

Input to template:

```python
{"country": "India"}
```

Output from `.format()`:

```python
"What is the capital of India?"
```

Output from `.invoke()`:

```python
StringPromptValue(...)
```

Both can be passed to `model.invoke()`.


In [ ]:
from langchain_core.prompts import PromptTemplate

capital_prompt = PromptTemplate.from_template("What is the capital of {country}?")

formatted_string = capital_prompt.format(country="India")
prompt_value = capital_prompt.invoke({"country": "India"})

print("Using .format():")
print(formatted_string)
print(type(formatted_string))

print("\nUsing .invoke():")
print(prompt_value)
print(type(prompt_value))

if model:
    print("\nModel response from formatted string:")
    print(model.invoke(formatted_string).content)

    print("\nModel response from PromptValue:")
    print(model.invoke(prompt_value).content)


### Function-by-function explanation

#### `PromptTemplate.from_template(...)`

- **Purpose:** create a reusable prompt template.
- **Input:** a string containing placeholders such as `{country}`.
- **Output:** a `PromptTemplate` object.
- **Internal logic:** stores the template and knows which variables must be provided.
- **Possible errors:** missing placeholder values during `.format()` or `.invoke()`.

#### `.format(country="India")`

- **Purpose:** directly convert template into a plain string.
- **Input:** keyword arguments matching placeholders.
- **Output:** `str`.
- **Good for:** simple scripts, printing/debugging prompt text.

#### `.invoke({"country": "India"})`

- **Purpose:** use the template as a LangChain `Runnable`.
- **Input:** dictionary.
- **Output:** `PromptValue`.
- **Good for:** chains such as `prompt | model`.

### Interview point

`.format()` is Python-style string formatting.

`.invoke()` is LangChain-style runnable execution.

---

## 5. Chat Prompts: `ChatPromptTemplate`

### What we are building

A multi-role chat prompt with a system instruction and user message.

### Background idea

Chat models understand roles:

| Role | Meaning |
|---|---|
| `system` | high-level behavior/instruction |
| `user` / `human` | user request |
| `assistant` / `ai` | previous model response |

A `ChatPromptTemplate` preserves these roles.

### Important gotcha

For chat prompts:

```python
chat_prompt.invoke({...})
```

is usually better than:

```python
chat_prompt.format(...)
```

because `.invoke()` keeps role structure, while `.format()` flattens everything into a single string.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate([
    ("system", "Reply in {language}. Keep the answer short."),
    ("user", "What is the capital of {country}?"),
])

# Correct for chat models: preserves system/user roles.
chat_prompt_value = chat_prompt.invoke({
    "language": "Hindi",
    "country": "India",
})

print("Messages produced by .invoke():")
print(chat_prompt_value.to_messages())

# Gotcha: this becomes one plain string.
flat_prompt = chat_prompt.format(language="Hindi", country="India")
print("\nFlattened prompt produced by .format():")
print(repr(flat_prompt))

if model:
    print("\nModel response using role-preserved prompt:")
    print(model.invoke(chat_prompt_value).content)

    print("\nModel response using flattened prompt:")
    print(model.invoke(flat_prompt).content)


### Why this matters

When you pass a real message list to a chat model, the model receives:

```text
system: Reply in Hindi. Keep the answer short.
user: What is the capital of India?
```

When you pass the flattened string, the model receives something closer to:

```text
human: System: Reply in Hindi. Keep the answer short.
Human: What is the capital of India?
```

It may still work, but it is not the same.

### Function-by-function explanation

#### `ChatPromptTemplate([...])`

- **Purpose:** create a multi-message prompt.
- **Input:** list of `(role, template)` pairs.
- **Output:** chat prompt template object.
- **Internal logic:** fills placeholders and produces role-specific messages.

#### `.invoke({...})`

- **Purpose:** fill placeholders and preserve message roles.
- **Output:** `ChatPromptValue`.
- **Useful method:** `.to_messages()` converts it into actual message objects.

#### `.format(...)`

- **Purpose:** flatten the whole chat prompt into one string.
- **Output:** `str`.
- **Risk:** system role is no longer a real system message.

---

## 6. Raw Messages: `SystemMessage` and `HumanMessage`

### What we are building

We manually create messages instead of using a template.

### Background idea

Under the hood, chat prompts become message objects. You can also create those objects yourself.

### When to use raw messages

Use raw messages when:

- you already have a conversation history,
- you need exact role control,
- you are building a chat app,
- you want to add previous assistant responses manually.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="Reply in Hindi. Keep it simple."),
    HumanMessage(content="What is the capital of India?"),
]

if model:
    response = model.invoke(messages)
    print(response.content)

# Conversation history example
conversation_history = [
    SystemMessage(content="You are a helpful tutor."),
    HumanMessage(content="Explain what caching means in one line."),
    AIMessage(content="Caching means saving a previous result so you can reuse it later without recomputing it."),
    HumanMessage(content="Now explain it with an LLM API example."),
]

if model:
    print(model.invoke(conversation_history).content)


### Important concept: LLMs do not remember automatically

The model has no memory of earlier calls unless you send the earlier messages again.

So for a chat app:

```python
history.append(HumanMessage(...))
response = model.invoke(history)
history.append(response)
```

That is how memory-like behavior is created.

### Interview answer

**Question:** How does LangChain maintain conversation memory?

**Answer:** The model itself does not remember. LangChain/user code resends previous messages as context, or uses a memory/retriever system to reconstruct relevant context.

---

## 7. Invocation Modes: `.invoke()`, `.batch()`, `.stream()`, `.astream_events()`

### What we are building

We learn the four important ways to run a model.

| Method | Input | Output | Use case |
|---|---|---|---|
| `.invoke(x)` | one input | one complete response | normal call |
| `.batch([x1, x2])` | list of inputs | list of complete responses | many independent prompts |
| `.stream(x)` | one input | chunks of text/message | live typing UI |
| `.astream_events(x)` | one input | async event dictionaries | debugging chains, custom event UI, tracing |

---

## 8. Batch Responses: `.batch()`

### What we are building

We send multiple independent prompts in one call.

### Background idea

`.batch()` is useful when prompts do not depend on each other.

Instead of:

```python
for prompt in prompts:
    model.invoke(prompt)
```

we use:

```python
model.batch(prompts)
```

LangChain can run them concurrently where possible.

### Input and output

Input:

```python
["Prompt 1", "Prompt 2", "Prompt 3"]
```

Output:

```python
[AIMessage(...), AIMessage(...), AIMessage(...)]
```

The output order matches the input order.


In [ ]:
prompts = [
    "What is the capital of India?",
    "What is the capital of Pakistan?",
    "What is the capital of the United States?",
]

if model:
    responses = model.batch(prompts)

    for prompt, response in zip(prompts, responses):
        print("PROMPT:", prompt)
        print("ANSWER:", response.content)
        print("-" * 60)


### Function-by-function explanation

#### `.batch(prompts)`

- **Purpose:** process many independent inputs.
- **Input:** list of strings/messages/prompt values.
- **Output:** list of `AIMessage` objects.
- **Internal logic:** LangChain schedules multiple calls, often concurrently.
- **Possible errors:** rate limits, one bad prompt shape, network failures.

### When to use `.batch()`

Good:

- generate answers for 100 questions,
- classify many reviews,
- extract fields from many documents,
- run the same chain over many inputs.

Not good:

- when prompt 2 depends on prompt 1,
- when you need token-by-token UI,
- when the provider has strict rate limits and you need careful throttling.

---

## 9. Streaming: `.stream()`

### What we are building

We print the model output chunk by chunk.

### Background idea

Without streaming, the user waits until the full answer is done.

With streaming, the UI can show tokens/chunks as they arrive.

This is why ChatGPT-like apps feel like they are typing.

### Input and output

Input:

```python
"Tell me about the NBA"
```

Output:

```python
AIMessageChunk(...)
AIMessageChunk(...)
AIMessageChunk(...)
```

Each chunk contains partial content.


In [ ]:
if model:
    for chunk in model.stream("Explain LangChain streaming in one short paragraph."):
        print(chunk.content, end="", flush=True)


### Important print parameters

```python
print(chunk.content, end="", flush=True)
```

| Parameter | Meaning |
|---|---|
| `end=""` | do not add a new line after every chunk |
| `flush=True` | print immediately instead of waiting for buffer |

### Interview answer

**Question:** What is the difference between `.invoke()` and `.stream()`?

**Answer:** `.invoke()` waits and returns the final full response. `.stream()` returns partial chunks as the model generates them, useful for real-time user interfaces.

---

## 10. Async Event Streaming: `.astream_events()`

### What we are building

We read model lifecycle events asynchronously.

### Background idea

`.stream()` gives mostly generated content chunks.

`.astream_events()` gives a richer event stream. It tells you what is happening inside the runnable lifecycle.

For a chat model, events may include:

| Event | Meaning |
|---|---|
| `on_chat_model_start` | model call started |
| `on_chat_model_stream` | one output chunk was produced |
| `on_chat_model_end` | model call ended |

For chains, you can also see events from prompt templates, parsers, retrievers, tools, etc.

### Why async?

Streaming events happen over time. Python should not freeze while waiting for network chunks.

So we use:

```python
async for event in model.astream_events(...):
    ...
```

`async for` means: keep receiving events as they arrive.


In [ ]:
import asyncio

async def show_first_events(llm, prompt, max_events=6):
    '''
    Purpose:
        Print the first few lifecycle events from a LangChain runnable.

    Inputs:
        llm        -> a LangChain chat model or runnable
        prompt     -> input prompt string
        max_events -> number of events to print before stopping

    Output:
        Nothing returned. It prints event names and event data keys.

    Internal logic:
        - Calls llm.astream_events(...)
        - Uses async for because events arrive over time
        - Stops after max_events to avoid huge output
    '''
    count = 0

    async for event in llm.astream_events(prompt, version="v2"):
        count += 1
        print("EVENT:", event.get("event"))
        print("NAME :", event.get("name"))
        print("DATA KEYS:", list(event.get("data", {}).keys()))
        print("-" * 60)

        if count >= max_events:
            break

if model:
    await show_first_events(model, "Describe the NBA in one sentence.", max_events=6)


### Very important notebook note

In Jupyter notebooks, you can usually use:

```python
await show_first_events(...)
```

In a normal `.py` script, use:

```python
asyncio.run(show_first_events(...))
```

Do **not** use `asyncio.run()` inside a notebook if the notebook already has an active event loop.

### Function-by-function explanation

#### `async def show_first_events(...)`

- **Purpose:** define an asynchronous function.
- **Why async:** event chunks arrive over network time; async lets Python wait without blocking everything.
- **Output:** prints event information.

#### `llm.astream_events(prompt, version="v2")`

- **Purpose:** returns an async generator of event dictionaries.
- **Input:** prompt and event schema version.
- **Output:** event dicts.
- **Useful fields:** `event`, `name`, `data`, `run_id`, `parent_ids`, `tags`, `metadata`.

#### `async for event in ...`

- **Purpose:** iterate over values that arrive asynchronously.
- **Common error:** using normal `for` instead of `async for`.

### `.stream()` vs `.astream_events()`

| Feature | `.stream()` | `.astream_events()` |
|---|---|---|
| Gives generated text chunks | yes | yes, inside event data |
| Gives lifecycle events | no | yes |
| Good for simple typing UI | yes | possible, but more complex |
| Good for debugging chains | limited | excellent |
| Requires async handling | no | yes |

---

## 11. Caching Responses

### What we are building

We avoid repeating identical LLM calls.

### Background idea

LLM calls cost time and sometimes money. If the same prompt and same model configuration are used again, cache can return the old response immediately.

LangChain supports global cache configuration:

```python
set_llm_cache(...)
```

Two cache types from the scripts:

| Cache | Storage | Survives Python restart? | Use case |
|---|---|---|---|
| `InMemoryCache()` | RAM | no | quick testing |
| `SQLiteCache("cache.db")` | disk database | yes | reuse cache across runs |

### Important concept

Cache key is not only prompt text. It includes model/config serialization too.

So this can be a cache miss:

```python
same prompt + temperature=0
same prompt + temperature=0.8
```


In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_community.cache import SQLiteCache
import time

# In-memory cache: disappears after the Python process stops.
set_llm_cache(InMemoryCache())

if model:
    prompt = "Explain caching in one sentence."

    start = time.time()
    response_1 = model.invoke(prompt)  # likely cache miss
    response_2 = model.invoke(prompt)  # likely cache hit
    end = time.time()

    print("First response:", response_1.content)
    print("Second response:", response_2.content)
    print("First usage metadata:", response_1.usage_metadata)
    print("Second usage metadata:", response_2.usage_metadata)
    print("Total time:", round(end - start, 3), "seconds")


In [ ]:
# SQLite cache: survives across restarts.
# Use this when you want repeated scripts/notebooks to reuse older results.

set_llm_cache(SQLiteCache(database_path="cache_response.db"))

if model:
    prompt = "Explain SQLite cache in one sentence."
    print(model.invoke(prompt).content)


### Function-by-function explanation

#### `set_llm_cache(...)`

- **Purpose:** sets LangChain's global cache backend.
- **Input:** cache object such as `InMemoryCache()` or `SQLiteCache(...)`.
- **Output:** no direct output; it changes global LangChain behavior.
- **Possible error:** forgetting to call it before model calls means no caching.

#### `InMemoryCache()`

- **Purpose:** save results in Python memory.
- **Input:** none.
- **Output:** cache backend object.
- **Limitation:** deleted when process ends.

#### `SQLiteCache(database_path="cache_response.db")`

- **Purpose:** save responses in a `.db` file.
- **Input:** database path.
- **Output:** cache backend object.
- **Advantage:** persistent between runs.

### How to detect cache hit

A common signal: usage metadata may be empty or missing because no real generation happened.

```python
response.usage_metadata
```

But do not rely only on this across every provider/version. Timing and cache database inspection can also help.

---

## 12. Token Tracking and Metadata

### What we are building

We inspect how many tokens a call used.

### Background idea

Token usage matters because:

- API cost is token-based,
- long prompts can become expensive,
- output length affects latency,
- context windows have limits.

### Two places to inspect tokens

| Place | Format | Why useful |
|---|---|---|
| `response.usage_metadata` | LangChain-normalized | portable across providers |
| `response.response_metadata["token_usage"]` | provider-specific | may include timings and raw fields |


In [ ]:
if model:
    prompt = PromptTemplate.from_template("Explain {topic} in two bullet points.")
    response = model.invoke(prompt.invoke({"topic": "LangChain token tracking"}))

    print("ANSWER:")
    print(response.content)

    print("\nNormalized usage_metadata:")
    print(response.usage_metadata)

    print("\nProvider response_metadata token_usage:")
    print(response.response_metadata.get("token_usage"))


### Callback-based tracking

If you are making many calls, manually checking every response is annoying.

LangChain callbacks can aggregate usage across a block of code.


In [ ]:
from langchain_community.callbacks import get_usage_metadata_callback

if model:
    with get_usage_metadata_callback() as cb:
        model.invoke("Explain prompts in one sentence.")
        model.invoke("Explain messages in one sentence.")
        model.invoke("Explain caching in one sentence.")

    print(cb.usage_metadata)


### Function-by-function explanation

#### `response.usage_metadata`

- **Purpose:** get normalized token counts.
- **Typical fields:** `input_tokens`, `output_tokens`, `total_tokens`.
- **Good for:** provider-independent code.

#### `response.response_metadata["token_usage"]`

- **Purpose:** get raw provider token usage.
- **Possible fields:** `prompt_tokens`, `completion_tokens`, `total_tokens`, timing fields.
- **Good for:** provider-specific debugging.

#### `get_usage_metadata_callback()`

- **Purpose:** aggregate token usage across multiple calls.
- **Input:** used as a context manager.
- **Output:** callback object with totals after the block.

### Common errors

| Error | Reason | Fix |
|---|---|---|
| `KeyError: 'token_usage'` | provider did not return that field | use `.get("token_usage")` |
| usage is empty | cache hit or provider behavior | disable cache while measuring cost |
| cost estimate wrong | pricing differs by model/provider | check latest provider pricing manually |

---

## 13. Structured Output: Raw JSON Schema

### What we are building

We force the model to return a Python dictionary with specific fields.

### Background idea

Normal LLM output is free text. But applications often need structured data:

```python
{
    "name": "...",
    "country": "...",
    "age": 42
}
```

LangChain provides:

```python
model.with_structured_output(schema)
```

Internally, this usually uses tool/function calling where the model fills arguments matching your schema.

### Why temperature should be low

Structured extraction should be stable. Use:

```python
temperature=0
```

not high creativity.


In [ ]:
country_schema = {
    "name": "country_info_schema",
    "description": "Extract basic information about a country from the user request.",
    "parameters": {
        "type": "object",
        "properties": {
            "country": {
                "type": "string",
                "description": "Name of the country mentioned by the user."
            },
            "capital": {
                "type": ["string", "null"],
                "description": "Capital city if known, otherwise null."
            },
            "continent": {
                "type": ["string", "null"],
                "description": "Continent if known, otherwise null."
            },
        },
        "required": ["country", "capital", "continent"],
    },
}

if api_key:
    structured_model = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=api_key,
        temperature=0,
    ).with_structured_output(country_schema)

    result = structured_model.invoke("Give basic info about India.")
    print(result)
    print(type(result))


### Function-by-function explanation

#### Schema dictionary

- **Purpose:** define exactly what fields the model should return.
- **Input:** JSON-schema-like Python dictionary.
- **Output:** used by LangChain to configure structured generation.

Important schema parts:

| Key | Meaning |
|---|---|
| `name` | tool/schema name |
| `description` | tells the model what the schema is for |
| `parameters` | object structure |
| `properties` | fields to extract/generate |
| `required` | fields that must appear |
| `type: ["string", "null"]` | value can be string or null |

#### `.with_structured_output(schema)`

- **Purpose:** creates a new runnable model that returns structured data.
- **Input:** schema dict or Pydantic model.
- **Output:** a structured-output runnable.
- **Return type with dict schema:** usually `dict`.

### Important warning about internet/search prompts

A system prompt like:

```text
Search the internet deeply.
```

does **not** give the model internet access.

For real search, you must provide a real search tool/API. Prompt text alone cannot create new capabilities.

---

## 14. Structured Output: Pydantic Model

### What we are building

We define the schema using a Python class instead of a raw dictionary.

### Background idea

Pydantic validates data and gives cleaner typed objects.

Instead of:

```python
result["country"]
```

we can use:

```python
result.country
```

### Why Pydantic is preferred

| Raw schema dict | Pydantic class |
|---|---|
| verbose | clean Python syntax |
| returns dict | returns typed object |
| less IDE support | better autocomplete |
| easy to make schema mistakes | validation is clearer |


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class CountryInfo(BaseModel):
    '''Basic information about a country.'''

    country: str = Field(..., description="Name of the country")
    capital: Optional[str] = Field(..., description="Capital city, or None if unknown")
    continent: Optional[str] = Field(..., description="Continent, or None if unknown")

if api_key:
    pydantic_structured_model = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=api_key,
        temperature=0,
    ).with_structured_output(CountryInfo)

    result = pydantic_structured_model.invoke("Give basic info about Japan.")
    print(result)
    print(result.country)
    print(result.capital)
    print(type(result))


### Function-by-function explanation

#### `class CountryInfo(BaseModel)`

- **Purpose:** define the expected output shape.
- **Input:** class fields and types.
- **Output:** Pydantic model class.
- **Internal logic:** Pydantic validates returned data against field types.

#### `Field(..., description="...")`

- **Purpose:** add requirement and description for a field.
- `...` means the field is required.
- `description` helps the LLM understand what to put there.

#### `Optional[str]`

- **Purpose:** value may be `str` or `None`.
- Important: `Optional[str]` does not automatically mean the field can be missing.
- With `Field(...)`, the field is required, but its value can be `None`.

### Interview answer

**Question:** Difference between `Optional[int]` and `Field(...)`?

**Answer:** `Optional[int]` means the value can be an integer or `None`. `Field(...)` means the field itself is required in the output. Together, it means the key must exist, but the value may be null.

---

## 15. Runnable and LCEL Chaining

### What we are building

We connect prompt → model → parser into one pipeline.

### Background idea

LangChain components share a common interface called `Runnable`.

A runnable supports methods like:

```python
.invoke()
.batch()
.stream()
```

Because prompts and models are both runnables, they can be chained using `|`.

```python
chain = prompt | model
```

This is called LCEL: **LangChain Expression Language**.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

explain_prompt = PromptTemplate.from_template(
    "Explain {concept} in exactly three simple bullet points."
)

parser = StrOutputParser()

if model:
    chain = explain_prompt | model | parser

    answer = chain.invoke({"concept": "LangChain PromptTemplate"})
    print(answer)

    print("\nBatch through full chain:")
    outputs = chain.batch([
        {"concept": "ChatGroq"},
        {"concept": "streaming"},
        {"concept": "structured output"},
    ])

    for output in outputs:
        print("-" * 60)
        print(output)


### Function-by-function explanation

#### `StrOutputParser()`

- **Purpose:** convert `AIMessage` into plain string.
- **Input:** model response object.
- **Output:** `str`.
- **Why useful:** after parser, no need to write `.content`.

#### `prompt | model | parser`

- **Purpose:** build a pipeline.
- **Input:** dictionary for prompt variables.
- **Output:** final parsed string.
- **Internal logic:** output of each step becomes input of next step.

### Why LCEL matters

Without chain:

```python
prompt_value = prompt.invoke({...})
response = model.invoke(prompt_value)
text = response.content
```

With chain:

```python
chain = prompt | model | parser
text = chain.invoke({...})
```

Cleaner and easier to extend.

---

## 16. Common Patterns From All Scripts

### Pattern 1: Simple one-shot answer

```python
response = model.invoke("Where is the Statue of Liberty?")
print(response.content)
```

Use when you need one answer.

### Pattern 2: Template-based answer

```python
prompt = PromptTemplate.from_template("What is the capital of {country}?")
response = model.invoke(prompt.invoke({"country": "India"}))
print(response.content)
```

Use when some part of the prompt changes.

### Pattern 3: Role-based chat

```python
chat_prompt = ChatPromptTemplate([
    ("system", "Reply in {language}"),
    ("user", "What is {topic}?"),
])
messages = chat_prompt.invoke({"language": "Hindi", "topic": "LangChain"})
response = model.invoke(messages)
```

Use when behavior/system instruction matters.

### Pattern 4: Many independent calls

```python
responses = model.batch(prompts)
```

Use when you have many independent prompts.

### Pattern 5: Live UI

```python
for chunk in model.stream(prompt):
    print(chunk.content, end="", flush=True)
```

Use when user should see the answer as it is generated.

### Pattern 6: Debug event flow

```python
async for event in model.astream_events(prompt, version="v2"):
    print(event["event"])
```

Use when debugging chains/tools/retrievers/model lifecycle.

### Pattern 7: Cache repeated calls

```python
set_llm_cache(SQLiteCache(database_path="cache_response.db"))
```

Use when the same prompts repeat.

### Pattern 8: Structured extraction

```python
structured_model = model.with_structured_output(MySchema)
result = structured_model.invoke(text)
```

Use when you need dict/object output, not prose.

---

## 17. Important Library Methods Used

| Method / class | Simple meaning | What it returns |
|---|---|---|
| `load_dotenv()` | load variables from `.env` | boolean-ish success value |
| `os.getenv()` | read environment variable | string or `None` |
| `ChatGroq()` | create Groq chat model wrapper | chat model runnable |
| `model.invoke()` | run one model call | `AIMessage` |
| `model.batch()` | run many independent calls | list of `AIMessage` |
| `model.stream()` | stream chunks | generator of `AIMessageChunk` |
| `model.astream_events()` | stream async lifecycle events | async generator of event dicts |
| `PromptTemplate.from_template()` | create string prompt template | `PromptTemplate` |
| `prompt.format()` | fill template as plain string | `str` |
| `prompt.invoke()` | fill template as runnable | `PromptValue` |
| `ChatPromptTemplate()` | create role-based prompt | chat prompt template |
| `chat_prompt.invoke()` | fill and preserve roles | `ChatPromptValue` |
| `chat_prompt.format()` | fill but flatten roles | `str` |
| `SystemMessage()` | create system instruction | message object |
| `HumanMessage()` | create user message | message object |
| `AIMessage()` | represent assistant response/history | message object |
| `set_llm_cache()` | enable global LangChain cache | no direct response |
| `InMemoryCache()` | RAM cache | cache object |
| `SQLiteCache()` | disk-backed cache | cache object |
| `with_structured_output()` | force schema output | structured runnable |
| `BaseModel` | Pydantic schema base | class parent |
| `Field()` | define field requirements/description | field metadata |
| `StrOutputParser()` | convert AIMessage to string | parser runnable |
| `get_usage_metadata_callback()` | aggregate token usage | callback context manager |

---

## 18. Sanity Checks

Use these checks while debugging.

### Check 1: Is the model object created?

```python
print(model)
```

If it is `None`, your API key was not loaded.

### Check 2: What exactly is my prompt?

```python
print(prompt.format(country="India"))
```

### Check 3: Did chat roles survive?

```python
print(chat_prompt.invoke({...}).to_messages())
```

### Check 4: Did the response truncate?

```python
print(response.response_metadata.get("finish_reason"))
```

### Check 5: What was token usage?

```python
print(response.usage_metadata)
print(response.response_metadata.get("token_usage"))
```

### Check 6: Is caching active?

Run the same prompt twice and compare time/usage metadata.

---

## 19. Common Errors and Fixes

| Problem | Why it happens | Fix |
|---|---|---|
| API key not found | `.env` not loaded or wrong variable name | use `GROQ_API_KEY=...` and call `load_dotenv()` |
| `ModuleNotFoundError` | library missing | install requirements |
| `KeyError` in prompt | missing placeholder value | pass all required keys |
| Chat prompt ignores system role | used `.format()` instead of `.invoke()` | use `chat_prompt.invoke({...})` |
| Answer cut off | `max_tokens` too small | increase or remove `max_tokens` |
| Current factual answer is wrong | model has no live internet | use a real search tool/API |
| Structured output missing/wrong | schema unclear or temp too high | use clear fields, descriptions, `temperature=0` |
| `asyncio.run()` error in notebook | event loop already running | use `await function(...)` in notebook |
| Cache not working | prompt/config differs or cache not set early | set cache before calls and keep config same |
| Token metadata missing | provider/cache behavior | use `.get()` and disable cache for measurement |

---

## 20. Interview Cheat Sheet

### What is LangChain?

LangChain is a framework for building LLM applications. It provides common interfaces for prompts, chat models, output parsers, tools, retrievers, memory, streaming, callbacks, and chains.

### What is `ChatGroq`?

`ChatGroq` is LangChain's wrapper around Groq-hosted chat models. It lets Groq models behave like LangChain runnables.

### What does `.invoke()` return?

Usually an `AIMessage` containing the answer text plus metadata.

### Why use `PromptTemplate`?

To create reusable prompts with variable placeholders instead of manually formatting strings everywhere.

### Why use `ChatPromptTemplate`?

To preserve chat roles like system/user/assistant and build multi-message prompts cleanly.

### Difference between `.format()` and `.invoke()` on prompts?

`.format()` returns plain string. `.invoke()` returns LangChain prompt value and works naturally in chains.

### Difference between `.stream()` and `.astream_events()`?

`.stream()` gives output chunks. `.astream_events()` gives lifecycle event dictionaries, useful for debugging and tracing.

### Why use caching?

To avoid repeated API calls for identical prompt + model config, saving cost and time.

### Why use structured output?

Because many applications need reliable dictionaries/objects instead of free-form prose.

### Why use Pydantic?

It gives typed schema definitions, validation, dot-access, and cleaner structured output handling.

### Does telling the model to browse the internet work?

No. The model needs an actual search/browsing tool. A prompt instruction alone cannot create internet access.

---

## 21. How to Modify This Later

### To switch provider

Replace:

```python
from langchain_groq import ChatGroq
model = ChatGroq(...)
```

with another LangChain chat model wrapper, for example `ChatOpenAI`, while keeping prompts/chains mostly similar.

### To change the model

Change only:

```python
model="llama-3.3-70b-versatile"
```

### To make answers more creative

Increase:

```python
temperature=0.7
```

### To make output shorter

Use:

```python
max_tokens=100
```

or better, tell the model in the prompt:

```text
Answer in 3 bullet points.
```

### To add more structured fields

In Pydantic:

```python
class CountryInfo(BaseModel):
    country: str
    capital: Optional[str]
    population_estimate: Optional[int]
```

### To create a production pipeline

Move toward this pattern:

```python
prompt = ChatPromptTemplate([...])
model = ChatGroq(...)
parser = StrOutputParser()
chain = prompt | model | parser
```

Then use:

```python
chain.invoke({...})
chain.batch([...])
chain.stream({...})
```

---

## 22. Practice Exercises

1. Create a `PromptTemplate` that explains any ML concept in 5 lines.
2. Convert it to a chain using `prompt | model | StrOutputParser()`.
3. Run `.batch()` for `Linear Regression`, `Decision Tree`, and `Neural Network`.
4. Stream the answer for one concept using `.stream()`.
5. Create a Pydantic schema for extracting:
   - `concept_name`,
   - `definition`,
   - `use_case`,
   - `difficulty_level`.
6. Use SQLite caching and run the same prompt twice.
7. Print token usage after each call.

---

## Final Memory Hook

Remember this one flow:

```text
PromptTemplate / ChatPromptTemplate
        ↓
ChatGroq model
        ↓
AIMessage
        ↓
.content / metadata / structured output / parser
```

And remember these choices:

```text
one call          -> invoke
many calls        -> batch
live text         -> stream
debug lifecycle   -> astream_events
repeat prompts    -> cache
dict/object output -> structured output
```
